# Clock - label table, split, cleaning check (H1-H3)

**Spec:** PLAN.md H1-H3. **Runs on:** Colab + Google Drive ONLY (NHATS). Not run yet.

> NHATS data: do NOT use Colab's Gemini / any AI assistant in this notebook. Only aggregate counts are printed. The image grid in step 4 is for your eyes only - do not share it.

Put the repo's `common/` and `clock/` folders in `MyDrive/Alzheimers/` and the NHATS folder (image zips + `survey/`) in `MyDrive/Alzheimers/NHATS/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
ROOT = '/content/drive/MyDrive/Alzheimers'
NHATS = ROOT + '/NHATS'          # image zips here, survey/ inside
OUT = ROOT + '/clock_out'        # label_table.csv, split.csv are written here
import os; os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)
import numpy as np, pandas as pd
from clock import labels, split, preprocess

## 1. Label table (H1)
Expect 54,027 images indexed (Bone et al.).

In [ ]:
tab = labels.build_label_table(NHATS, NHATS + '/survey')
labels.summarize(tab)

## 2. Split by person (H2)
Each SPID in exactly one of train / val / test. The last line must print 0.

In [ ]:
tab = split.split_by_spid(tab, val_images=1000, test_frac=0.10, seed=0)
split.report(tab)
tab.to_csv(OUT + '/split.csv', index=False)   # stays on your Drive

## 3. Cleaning check (H3)
Shows 8 random clocks: original | after invert + name-box removal + rough crop. Look for (a) name boxes gone, (b) no clock strokes erased, (c) crop contains the whole clock. Tell Claude only what you saw, e.g. '7 of 8 fine, 1 clock lost its outline'.

In [ ]:
import zipfile, matplotlib.pyplot as plt
rows = tab[tab.split != 'excluded'].sample(8, random_state=1)
fig, ax = plt.subplots(8, 3, figsize=(9, 24))
for k, (_, r) in enumerate(rows.iterrows()):
    with zipfile.ZipFile(f'{NHATS}/{r.zip_name}') as z:
        img = preprocess.load_tif(z.read(r.member_path))
    inv = preprocess.invert(img)
    clean = preprocess.remove_name_boxes(inv)
    crop = preprocess.crop_to_mass(clean)
    for a, im, t in zip(ax[k], (img, clean, crop), ('original', 'boxes removed', 'rough crop')):
        a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()